# WP1.6 - Banked Transformer power grids Tests
This notebook demonstrates modeling of banked transformers and their integration within the power flow calculations. The tests serve as proof of concept for transformer modeling as individual single-phase transformers in a banked unit into pandapower multiconductor.
The proof is based on the tests defined in the WP1.6 specification document.
All results have been compared against openDSS to confirm their correctness.

Here the needed libraries and functions used for the tests are imported.

In [1]:
import multiconductor as mc
import multiconductor.test.testing_toolbox
import pandas as pd
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
pd.set_option('display.float_format', '{:.12f}'.format) 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
grids = multiconductor.test.testing_toolbox.load_all_test_grids("wp1_6_banked_transformer")
l = []
for name, net in grids.items():
    mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)
    c = multiconductor.test.testing_toolbox.compare_to_dss(net)
    l.append(c)
pd.DataFrame(l, index=grids.keys()).sort_index(key=lambda x: x.str.len())

,max_vm_pu_diff,max_va_degree_diff,max_i_from_ka_diff,max_i_to_ka_diff,max_p_from_mw_diff,max_p_to_mw_diff,max_q_from_mvar_diff,max_q_to_mvar_diff
two_bus_1ph_variant_0_p2n_gnd1_case1.pkl,0.000000006118,0.000002232179,NaN,NaN,NaN,NaN,NaN,NaN
two_bus_1ph_variant_0_p2n_gnd1_case2.pkl,0.000000454719,0.000016706403,NaN,NaN,NaN,NaN,NaN,NaN
two_bus_1ph_variant_0_p2p_gnd1_case1.pkl,0.000000012182,0.000001884601,NaN,NaN,NaN,NaN,NaN,NaN
two_bus_1ph_variant_0_p2n_gnd1_case3.pkl,0.000000479513,0.000016735299,NaN,NaN,NaN,NaN,NaN,NaN
two_bus_1ph_variant_0_p2p_gnd1_case3.pkl,0.000000392934,0.000016348337,NaN,NaN,NaN,NaN,NaN,NaN
two_bus_1ph_variant_0_p2p_gnd1_case2.pkl,0.000000365426,0.000016995718,NaN,NaN,NaN,NaN,NaN,NaN
five_bus_1ph_variant_0_p2n_gnd2_case1.pkl,0.000000012048,0.000002461165,0.000000002593,0.000000002593,0.000000000098,0.000000000093,0.000000000344,0.000000000325
five_bus_1ph_variant_0_p2n_gnd3_case1.pkl,0.000000011467,0.000002426395,0.000000002493,0.000000002493,0.000000000096,0.000000000090,0.000000000346,0.000000000327
five_bus_1ph_variant_0_p2p_gnd1_case2.pkl,0.000000455662,0.000019225780,0.000000088692,0.000000088692,0.000000003339,0.000000003428,0.000000004667,0.000000002723
five_bus_1ph_variant_0_p2p_gnd1_case1.pkl,0.000000015000,0.000002600568,0.000000003241,0.000000003241,0.000000000041,0.000000000021,0.000000000442,0.000000000544


## Tests on 2-bus grid
These tests serve to demonstrate the operation of 3 phase transformer as a banked unit of three 1 phase transformers in a simple 2 bus grid. The operation of grids with different configurations on low voltage and high voltage bus has been implemented such as: Wye-Wye (phase-to-neutral), Delta-Delta (phase-to-phase) and Delta-Wye (mixed). The tests for a simple 2 bus grid considers one grounding option. The following scheme shows the general grid topology, with delta at high voltage bus and wye at low voltage bus and grounding at bus 1 and 2. For each grid, 3 transformer test cases are implemented.

<img src="grid_images/two_bus_mixed.png" width="50%">

Transformer Settings:
1. vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral
2. vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral
3. vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

**Test Cases Scheme**

| Variant | LV bus | HV bus | Grounding opt | Transformer Settings |
|:------:|:------:|:------:|:-------------:|:---------:|
|   0    |  p2n  |  p2n   |      1        | (1, 2, 3) |
|   0    |  p2p   |  p2p   |      —        | (1, 2, 3) |
|   0    |  p2p   |  p2n   |      —        | (1, 2, 3) |

- p2n : phase-to-neutral 
- p2p : phase-to-phase

### 2 bus grid, Delta (phase to phase on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral

In [7]:
net = grids[f"two_bus_1ph_variant_0_mixed_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 2.2610292971236845e-08
Max overall va_degree difference: 2.1865105637175475e-06
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000001933 1.000000001933   -0.000000713544   -0.000000694178
      2     0.999999997620 0.999999997620 -120.000000755261 -120.000000742350
      3     0.999999999146 0.999999999146  119.999999479573  119.999999486029
1     0     0.000000000000 0.000000000000 -175.367641819005    2.171787233975
      1     0.997160197671 0.997160220282   29.476606888793   29.476609075304
      2     0.998256627936 

### 2 bus grid, Delta (phase to phase on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [4]:
net = grids[f"two_bus_1ph_variant_0_mixed_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.547539639299103e-07
Max overall va_degree difference: 1.6703975337151178e-05
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000001930 1.000000001930   -0.000000735879   -0.000000705357
      2     0.999999997617 0.999999997617 -120.000000777542 -120.000000753476
      3     0.999999999144 0.999999999144  119.999999457273  119.999999474884
1     0     0.000000000000 0.000000000000   14.405557612299    2.159820631065
      1     0.997129354066 0.997128899312   29.470393374806   29.470410078781
      2     0.998225997301 0

### 2 bus grid, Delta (phase to phase on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [5]:
net = grids[f"two_bus_1ph_variant_0_mixed_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.79549550069791e-07
Max overall va_degree difference: 1.6732872541780353e-05
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000001930 1.000000001930   -0.000000735879   -0.000000705357
      2     0.999999997617 0.999999997617 -120.000000777542 -120.000000753476
      3     0.999999999144 0.999999999144  119.999999457273  119.999999474884
1     0     0.000000000000 0.000000000000 -138.481722683262    2.159654789023
      1     1.046985821769 1.046985342219   29.470393374806   29.470410107678
      2     1.048137297166 1.

### 2 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral
<img src="grid_images/two_bus_p2n.png" width="50%">

In [6]:
net = grids[f"two_bus_1ph_variant_0_p2n_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 6.1176564924991794e-09
Max overall va_degree difference: 2.232178957117803e-06
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000069 0.000000000000  -12.172592984102    0.000000000000
      1     0.999999999452 0.999999999452   -0.000000947825   -0.000000928458
      2     0.999999999615 0.999999999615 -120.000000568107 -120.000000555196
      3     0.999999999632 0.999999999632  119.999999526700  119.999999533155
1     0     0.000000000000 0.000000000000  104.037872692725  -27.828150969353
      1     0.997160213595 0.997160219713   -0.523393296341   -0.523391064162
      2     0.998256647016 0

### 2 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [7]:
net = grids[f"two_bus_1ph_variant_0_p2n_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.547187628656246e-07
Max overall va_degree difference: 1.6706403085042787e-05
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000069 0.000000000000  -12.185011310601    0.000000000000
      1     0.999999999448 0.999999999448   -0.000000970129   -0.000000939608
      2     0.999999999613 0.999999999613 -120.000000590413 -120.000000566346
      3     0.999999999630 0.999999999630  119.999999504394  119.999999522004
1     0     0.000000000000 0.000000000000  -89.998370769270  -27.840196275929
      1     0.997129353461 0.997128898742   -0.529606767081   -0.529590060678
      2     0.998225999742 0

### 2 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [8]:
net = grids[f"two_bus_1ph_variant_0_p2n_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.795125934098365e-07
Max overall va_degree difference: 1.6735299485426403e-05
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000069 0.000000000000  -12.185011310640    0.000000000000
      1     0.999999999448 0.999999999448   -0.000000970129   -0.000000939608
      2     0.999999999613 0.999999999613 -120.000000590413 -120.000000566346
      3     0.999999999630 0.999999999630  119.999999504394  119.999999522004
1     0     0.000000000000 0.000000000000   63.436426584316  -27.840330392009
      1     1.046985821134 1.046985341621   -0.529606767080   -0.529590031781
      2     1.048137299729 1

### 2 bus grid, Delta (phase to phase on hv bus)- Delta (phase to phase on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral
<img src="grid_images/two_bus_p2p.png" width="50%">

In [9]:
net = grids[f"two_bus_1ph_variant_0_p2p_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.2182135189497956e-08
Max overall va_degree difference: 1.8846011826112452e-06
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000001938 1.000000001938   -0.000000712515   -0.000000693148
      2     0.999999997638 0.999999997638 -120.000000755757 -120.000000742846
      3     0.999999999134 0.999999999134  119.999999479213  119.999999485669
1     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999165991184 0.999165995738   -0.413649930658   -0.413648046057
      2     0.996656992549 

### 2 bus grid, Delta (phase to phase on hv bus)- Delta (phase to phase on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [10]:
net = grids[f"two_bus_1ph_variant_0_p2p_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 3.654261987451335e-07
Max overall va_degree difference: 1.699571794233634e-05
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000001935 1.000000001935   -0.000000734849   -0.000000704328
      2     0.999999997635 0.999999997635 -120.000000778039 -120.000000753972
      3     0.999999999132 0.999999999132  119.999999456913  119.999999474524
1     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999135259435 0.999134894009   -0.419864854785   -0.419856101253
      2     0.996626243262 0.

### 2 bus grid, Delta (phase to phase on hv bus)- Delta (phase to phase on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [11]:
net = grids[f"two_bus_1ph_variant_0_p2p_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 3.929341043651391e-07
Max overall va_degree difference: 1.634833749619702e-05
Max overall i_from_ka difference: nan
Max overall i_to_ka difference: nan
Max overall p_from_mw difference: nan
Max overall p_to_mw difference: nan
Max overall q_from_mvar difference: nan
Max overall q_to_mvar difference: nan

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000001935 1.000000001935   -0.000000734849   -0.000000704328
      2     0.999999997635 0.999999997635 -120.000000778039 -120.000000753972
      3     0.999999999132 0.999999999132  119.999999456913  119.999999474524
1     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.049092022407 1.049091629473   -0.419864854788   -0.419855491354
      2     1.046457555425 1.

## Tests on 5-bus grid
These tests serve to demonstrate the operation of 3 phase transformer as a banked unit of three 1 phase transformers in a simple 5 bus grid. The operation of grids with different configurations on low voltage and high voltage bus has been implemented such as: Wye-Wye (phase-to-neutral), Delta-Delta (phase-to-phase) and Delta-Wye (mixed). For phase to neutral and mixed configurations, 3 grounding options have been implemented as mentioned in the table below. The 5 bus tests include 3 transformer settings for each transformer configuration.

<img src="grid_images/five_bus_p2n_gnd1.png" width="50%">

 **Grounding Option for Variant 0**
 
 G = grounded, u = ungrounded

| Connection | Grounding Option  | Bus1  | Bus2 | Bus3  | Bus4  | Bus5  |
|:----------:|:-------:|:------:|:------:|:------:|:------:|:------:|
| p2n        |   1     |   G    |   G    |   u    |   u    |   u    |
| p2n        |   2     |   G    |   G    |   G    |   u    |   u    |
| p2n        |   3    |   G    |   G    |   G    |   G    |   G    |
| p2p        |   1    |   G    |   G    |   u    |   u    |   u    |
| mixed      |   1     |   G    |   G    |   u    |   u    |   u    |
| mixed      |   2     |   G    |   G    |   G    |   u    |   u    |
| mixed      |   3  |   G    |   G    |   G    |   G    |   G    |


- **G = grounded neutral (phase 0)**
- p2n: phase-to-neutral
- p2p: phase-to-phase
- mixed: phase-to-phase on high voltage bus and phase-to-neutral on low volatge bus

Transformer Settings:
1. vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral
2. vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral
3. vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

**5-Bus Test Scheme (Variant 0)**

| Connection | Grounding option | Line 2–3 | Line 3–4 | Line 3–5 | Transformer Settings    |
|:----------:|:----------------:|:--------:|:--------:|:--------:|:---------:|
| p2n        | 1 / 2 / 3        | **3φ**   | **1φ**   | **1φ**   | (1, 2, 3) |
| p2p        | 1                | **3φ**   | **1φ**   | **1φ**   | (1, 2, 3) |
| mixed      | 1 / 2 / 3        | **3φ**   | **1φ**   | **1φ**   | (1, 2, 3) |


- **3ϕ (3ϕ+N)** four conductors 
- **1ϕ (single phase + neutral)** two conductors

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral

In [12]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.3111940422305679e-08
Max overall va_degree difference: 2.4902595479048273e-06
Max overall i_from_ka difference: 3.14551662494722e-09
Max overall i_to_ka difference: 3.14551662494722e-09
Max overall p_from_mw difference: 9.554368754494114e-11
Max overall p_to_mw difference: 9.02156196969095e-11
Max overall q_from_mvar difference: 3.3694968169795114e-10
Max overall q_to_mvar difference: 3.1830143425518043e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000090 0.000000000000  -57.013203067560    0.000000000000
      1     0.999999996728 0.999999996728   -0.000001045604   -0.000001026237
      2     0.999999998641 0.999999998641 -120.000000953252 -120.000000940340
      3     0.999999998567 0.999999998567  119.999999469818  119.999999476273
1     0     0.000000000000 0.000000000000  -97.123854523311  -72.66852256547

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [13]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.866520939561525e-07
Max overall va_degree difference: 2.2661825627379173e-05
Max overall i_from_ka difference: 1.1970990590470798e-07
Max overall i_to_ka difference: 1.1970990590470798e-07
Max overall p_from_mw difference: 4.688543029512937e-09
Max overall p_to_mw difference: 4.44960025075547e-09
Max overall q_from_mvar difference: 7.995131090163254e-09
Max overall q_to_mvar difference: 3.917802961390937e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000090 0.000000000000  -57.025823815720    0.000000000000
      1     0.999999996724 0.999999996724   -0.000001067878   -0.000001037356
      2     0.999999998638 0.999999998638 -120.000000975546 -120.000000951480
      3     0.999999998565 0.999999998565  119.999999447524  119.999999465135
1     0     0.000000000000 0.000000000000   90.001161829545  -72.6811543145

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [14]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 5.098023216909553e-07
Max overall va_degree difference: 2.226928007931228e-05
Max overall i_from_ka difference: 1.1216529502422468e-07
Max overall i_to_ka difference: 1.1216529502422468e-07
Max overall p_from_mw difference: 4.6154630083117265e-09
Max overall p_to_mw difference: 4.399866682797127e-09
Max overall q_from_mvar difference: 7.219432697491701e-09
Max overall q_to_mvar difference: 3.605650268807825e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000089 0.000000000000  -56.703585461770    0.000000000000
      1     0.999999996935 0.999999996935   -0.000001062680   -0.000001032159
      2     0.999999998779 0.999999998778 -120.000000975099 -120.000000951033
      3     0.999999998619 0.999999998619  119.999999448523  119.999999466133
1     0     0.000000000000 0.000000000000   14.865281804200  -72.358849801

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 2) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral

In [15]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd2_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.2047960296079907e-08
Max overall va_degree difference: 2.4611652378325743e-06
Max overall i_from_ka difference: 2.593046471988103e-09
Max overall i_to_ka difference: 2.593046471988103e-09
Max overall p_from_mw difference: 9.81326339632993e-11
Max overall p_to_mw difference: 9.273203038784317e-11
Max overall q_from_mvar difference: 3.4403103976732297e-10
Max overall q_to_mvar difference: 3.2512894503414214e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000088 0.000000000000  -59.337790885562    0.000000000000
      1     0.999999997034 0.999999997034   -0.000001021580   -0.000001002214
      2     0.999999998776 0.999999998776 -120.000000970420 -120.000000957508
      3     0.999999998567 0.999999998567  119.999999470174  119.999999476630
1     0     0.000000054802 0.000000054802  120.662360461357  120.662362969

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 2) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [16]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd2_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.7511228340457734e-07
Max overall va_degree difference: 2.1795204315666616e-05
Max overall i_from_ka difference: 1.088318437314495e-07
Max overall i_to_ka difference: 1.088318437314495e-07
Max overall p_from_mw difference: 4.692084391161311e-09
Max overall p_to_mw difference: 4.453068316867537e-09
Max overall q_from_mvar difference: 6.401816974102581e-09
Max overall q_to_mvar difference: 2.7880386818668335e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000088 0.000000000000  -59.350567957536    0.000000000000
      1     0.999999997030 0.999999997030   -0.000001043856   -0.000001013335
      2     0.999999998772 0.999999998772 -120.000000992717 -120.000000968651
      3     0.999999998565 0.999999998565  119.999999447881  119.999999465491
1     0     0.000000054804 0.000000054804  120.655783308300  120.655798957

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 2) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [17]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd2_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.993957789523051e-07
Max overall va_degree difference: 2.151902252456317e-05
Max overall i_from_ka difference: 1.030662386825032e-07
Max overall i_to_ka difference: 1.030662386825032e-07
Max overall p_from_mw difference: 4.64437692124231e-09
Max overall p_to_mw difference: 4.428197603739692e-09
Max overall q_from_mvar difference: 5.897220545225679e-09
Max overall q_to_mvar difference: 2.650444259032214e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000087 0.000000000000  -58.798063574744    0.000000000000
      1     0.999999997195 0.999999997195   -0.000001041339   -0.000001010818
      2     0.999999998905 0.999999998905 -120.000000990406 -120.000000966340
      3     0.999999998619 0.999999998619  119.999999448811  119.999999466421
1     0     0.000000051910 0.000000051910  121.208286187763  121.208302673123


### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 3) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral

In [18]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd3_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.1467482408100693e-08
Max overall va_degree difference: 2.4263952536074385e-06
Max overall i_from_ka difference: 2.493057565944312e-09
Max overall i_to_ka difference: 2.493057565944312e-09
Max overall p_from_mw difference: 9.559123631541766e-11
Max overall p_to_mw difference: 9.024487407360837e-11
Max overall q_from_mvar difference: 3.461098690635267e-10
Max overall q_to_mvar difference: 3.2739762160019503e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000087 0.000000000000  -58.266496101505    0.000000000000
      1     0.999999997269 0.999999997269   -0.000001017964   -0.000000998597
      2     0.999999998932 0.999999998932 -120.000000967961 -120.000000955050
      3     0.999999998568 0.999999998568  119.999999470511  119.999999476966
1     0     0.000000054168 0.000000054168  121.733579571886  121.733582066

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 3) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [19]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd3_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.747521712422653e-07
Max overall va_degree difference: 2.1359360569128683e-05
Max overall i_from_ka difference: 1.0687104784468104e-07
Max overall i_to_ka difference: 1.0687104784468104e-07
Max overall p_from_mw difference: 4.678313937633849e-09
Max overall p_to_mw difference: 4.439562870106606e-09
Max overall q_from_mvar difference: 5.55877667680893e-09
Max overall q_to_mvar difference: 2.029627046724186e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000087 0.000000000000  -58.279202776570    0.000000000000
      1     0.999999997265 0.999999997265   -0.000001040242   -0.000001009721
      2     0.999999998928 0.999999998928 -120.000000990260 -120.000000966194
      3     0.999999998566 0.999999998566  119.999999448217  119.999999465828
1     0     0.000000054170 0.000000054170  121.727072815737  121.7270891319

### 5 bus grid, Wye (phase to neutral on hv bus)- Wye (phase to neutral on lv bus), grounding at hv and lv bus (grounding option: 2) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [20]:
net = grids[f"five_bus_1ph_variant_0_p2n_gnd3_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.993055575663874e-07
Max overall va_degree difference: 2.1130346970044123e-05
Max overall i_from_ka difference: 1.016741528192533e-07
Max overall i_to_ka difference: 1.016741528192533e-07
Max overall p_from_mw difference: 4.6597177123797184e-09
Max overall p_to_mw difference: 4.443168995921232e-09
Max overall q_from_mvar difference: 5.166084637332757e-09
Max overall q_to_mvar difference: 1.978672188335945e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000086 0.000000000000  -57.832180457141    0.000000000000
      1     0.999999997404 0.999999997404   -0.000001038106   -0.000001007585
      2     0.999999999044 0.999999999044 -120.000000988203 -120.000000964137
      3     0.999999998620 0.999999998619  119.999999449086  119.999999466696
1     0     0.000000051380 0.000000051380  122.174093922483  122.1741112807

### 5 bus grid, Delta (phase to phase on hv bus)- Delta (phase to phase on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:1 vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral

In [21]:
net = grids[f"five_bus_1ph_variant_0_p2p_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.5000297337586233e-08
Max overall va_degree difference: 2.600568080879384e-06
Max overall i_from_ka difference: 3.241099305606454e-09
Max overall i_to_ka difference: 3.241099305606454e-09
Max overall p_from_mw difference: 4.1257657013016313e-11
Max overall p_to_mw difference: 2.1213912571038662e-11
Max overall q_from_mvar difference: 4.4236920463541773e-10
Max overall q_to_mvar difference: 5.435885162136112e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999998060 0.999999998060   -0.000000967684   -0.000000948317
      2     0.999999996513 0.999999996513 -120.000000757498 -120.000000744586
      3     1.000000000464 1.000000000464  119.999999214162  119.999999220618
1     0     0.000000000000 0.000000000000    0.000000000000    0.00000000

### 5 bus grid, Delta (phase to phase on hv bus)- Delta (phase to phase on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case:2 vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral

In [22]:
net = grids[f"five_bus_1ph_variant_0_p2p_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.5566188200396596e-07
Max overall va_degree difference: 1.9225780025888817e-05
Max overall i_from_ka difference: 8.869208220629865e-08
Max overall i_to_ka difference: 8.869208220629865e-08
Max overall p_from_mw difference: 3.3393448700680395e-09
Max overall p_to_mw difference: 3.428240545610972e-09
Max overall q_from_mvar difference: 4.666604384168105e-09
Max overall q_to_mvar difference: 2.723351589378753e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999998056 0.999999998056   -0.000000989972   -0.000000959450
      2     0.999999996510 0.999999996510 -120.000000779766 -120.000000755700
      3     1.000000000461 1.000000000461  119.999999191844  119.999999209455
1     0     0.000000000000 0.000000000000    0.000000000000    0.000000000

### 5 bus grid, Delta (phase to phase on hv bus)- Delta (phase to phase on lv bus), grounding at hv and lv bus (grounding option: 1) and asymmetric load at lv bus, case: 3 vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

In [23]:
net = grids[f"five_bus_1ph_variant_0_p2p_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.7217729637338834e-07
Max overall va_degree difference: 1.8906230513038302e-05
Max overall i_from_ka difference: 8.420964384137086e-08
Max overall i_to_ka difference: 8.420964384137086e-08
Max overall p_from_mw difference: 3.1808011086154053e-09
Max overall p_to_mw difference: 3.086845563882168e-09
Max overall q_from_mvar difference: 4.2991889076526846e-09
Max overall q_to_mvar difference: 2.8937040688972737e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999998165 0.999999998165   -0.000000988090   -0.000000957569
      2     0.999999996603 0.999999996603 -120.000000779117 -120.000000755051
      3     1.000000000543 1.000000000543  119.999999193941  119.999999211552
1     0     0.000000000000 0.000000000000    0.000000000000    0.0000000

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd1_case1

In [24]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.3094943573932483e-08
Max overall va_degree difference: 2.4911884324296807e-06
Max overall i_from_ka difference: 3.1434344016645355e-09
Max overall i_to_ka difference: 3.1434344016645355e-09
Max overall p_from_mw difference: 9.512529999811115e-11
Max overall p_to_mw difference: 8.980273816239226e-11
Max overall q_from_mvar difference: 3.3342717042222914e-10
Max overall q_to_mvar difference: 3.147982326201504e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000244 1.000000000244   -0.000000818307   -0.000000798941
      2     0.999999997219 0.999999997219 -120.000000967780 -120.000000954869
      3     0.999999996473 0.999999996473  119.999999257049  119.999999263505
1     0     0.000000000000 0.000000000000   75.594943647112  -42.6685966

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd1_case2

In [25]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.866690599403256e-07
Max overall va_degree difference: 2.2662894167524428e-05
Max overall i_from_ka difference: 1.197119517681866e-07
Max overall i_to_ka difference: 1.197119517681866e-07
Max overall p_from_mw difference: 4.688961059706731e-09
Max overall p_to_mw difference: 4.450012774936951e-09
Max overall q_from_mvar difference: 7.995331183577314e-09
Max overall q_to_mvar difference: 3.917933373565052e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000241 1.000000000241   -0.000000840625   -0.000000810104
      2     0.999999997216 0.999999997216 -120.000000990057 -120.000000965991
      3     0.999999996470 0.999999996470  119.999999234783  119.999999252393
1     0     0.000000000000 0.000000000000  -26.400118910685  -42.68111291026

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd1_case3

In [26]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 5.0982091948093e-07
Max overall va_degree difference: 2.2270289658621323e-05
Max overall i_from_ka difference: 1.1216962367277539e-07
Max overall i_to_ka difference: 1.1216962367277539e-07
Max overall p_from_mw difference: 4.614817056269871e-09
Max overall p_to_mw difference: 4.399205472127576e-09
Max overall q_from_mvar difference: 7.220107315838997e-09
Max overall q_to_mvar difference: 3.6061854396737814e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000353 1.000000000353   -0.000000834926   -0.000000804404
      2     0.999999997416 0.999999997416 -120.000000988402 -120.000000964336
      3     0.999999996564 0.999999996564  119.999999234071  119.999999251682
1     0     0.000000000000 0.000000000000  -92.526299311346  -42.3589076522

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd2_case1

In [27]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd2_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.2034951257788862e-08
Max overall va_degree difference: 2.4621103591471183e-06
Max overall i_from_ka difference: 2.591196868184653e-09
Max overall i_to_ka difference: 2.591196868184653e-09
Max overall p_from_mw difference: 9.774740739043608e-11
Max overall p_to_mw difference: 9.235239309401955e-11
Max overall q_from_mvar difference: 3.406163589028677e-10
Max overall q_to_mvar difference: 3.2173383098313413e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000278 1.000000000278   -0.000000801052   -0.000000781685
      2     0.999999997647 0.999999997647 -120.000000967196 -120.000000954285
      3     0.999999996451 0.999999996451  119.999999246422  119.999999252877
1     0     0.000000054802 0.000000054802  150.662360280374  150.662362786

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd2_case2

In [28]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd2_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.7512527268089855e-07
Max overall va_degree difference: 2.179628931386901e-05
Max overall i_from_ka difference: 1.0883368881109412e-07
Max overall i_to_ka difference: 1.0883368881109412e-07
Max overall p_from_mw difference: 4.692469631611962e-09
Max overall p_to_mw difference: 4.453447961100254e-09
Max overall q_from_mvar difference: 6.402118295570358e-09
Max overall q_to_mvar difference: 2.788278750248674e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000275 1.000000000275   -0.000000823368   -0.000000792847
      2     0.999999997643 0.999999997643 -120.000000989479 -120.000000965413
      3     0.999999996449 0.999999996448  119.999999224155  119.999999241765
1     0     0.000000054804 0.000000054804  150.655783127306  150.655798775

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd2_case3

In [29]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd2_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.994098481425624e-07
Max overall va_degree difference: 2.152005045985561e-05
Max overall i_from_ka difference: 1.030710021499015e-07
Max overall i_to_ka difference: 1.030710021499015e-07
Max overall p_from_mw difference: 4.6431215296183215e-09
Max overall p_to_mw difference: 4.42692004234968e-09
Max overall q_from_mvar difference: 5.897347677905063e-09
Max overall q_to_mvar difference: 2.6504213528760756e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000377 1.000000000377   -0.000000819806   -0.000000789285
      2     0.999999997794 0.999999997793 -120.000000987599 -120.000000963534
      3     0.999999996549 0.999999996549  119.999999224471  119.999999242081
1     0     0.000000051910 0.000000051910  151.208286561434  151.20830249274

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd3_case1

In [30]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd3_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.145370609467733e-08
Max overall va_degree difference: 2.4272666188096537e-06
Max overall i_from_ka difference: 2.491159029061052e-09
Max overall i_to_ka difference: 2.491159029061052e-09
Max overall p_from_mw difference: 9.524305996655436e-11
Max overall p_to_mw difference: 8.990228353433771e-11
Max overall q_from_mvar difference: 3.42655574053774e-10
Max overall q_to_mvar difference: 3.239628378927384e-10

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000379 1.000000000379   -0.000000795209   -0.000000775843
      2     0.999999997849 0.999999997848 -120.000000965458 -120.000000952546
      3     0.999999996541 0.999999996541  119.999999245253  119.999999251709
1     0     0.000000054168 0.000000054168  151.733579394791  151.733581888199

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd3_case2

In [31]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd3_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.7476592834883036e-07
Max overall va_degree difference: 2.1360346639909267e-05
Max overall i_from_ka difference: 1.0687294277333947e-07
Max overall i_to_ka difference: 1.0687294277333947e-07
Max overall p_from_mw difference: 4.678662141738288e-09
Max overall p_to_mw difference: 4.439905498809793e-09
Max overall q_from_mvar difference: 5.559018453893394e-09
Max overall q_to_mvar difference: 2.0298062566696806e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000376 1.000000000376   -0.000000817527   -0.000000787005
      2     0.999999997845 0.999999997845 -120.000000987743 -120.000000963677
      3     0.999999996538 0.999999996538  119.999999222985  119.999999240595
1     0     0.000000054170 0.000000054170  151.727072638632  151.7270889

### 5 bus grid, five_bus_1ph_variant_0_mixed_gnd3_case3

In [32]:
net = grids[f"five_bus_1ph_variant_0_mixed_gnd3_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 4.993206559333885e-07
Max overall va_degree difference: 2.1131292225362586e-05
Max overall i_from_ka difference: 1.0167731914756395e-07
Max overall i_to_ka difference: 1.0167731914756395e-07
Max overall p_from_mw difference: 4.659237020504525e-09
Max overall p_to_mw difference: 4.4426760985316616e-09
Max overall q_from_mvar difference: 5.1664542886908915e-09
Max overall q_to_mvar difference: 1.9789425866230392e-09

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     1.000000000467 1.000000000467   -0.000000814609   -0.000000784087
      2     0.999999997973 0.999999997973 -120.000000986028 -120.000000961962
      3     0.999999996629 0.999999996629  119.999999223413  119.999999241024
1     0     0.000000051380 0.000000051380  152.174094533911  152.174111

Transformer Settings:
1. vk/vkr > 0, NLL & i0 = 0, tap_pos = tap_neutral
2. vk/vkr > 0, NLL>0 & i0>0, tap_pos = tap_neutral
3. vk/vkr > 0, NLL>0 & i0>0, tap_pos != tap_neutral

### Variant 2 — Two 1ph transformers

| Connection | Grounding                           | Line 1–2 | Line 2–3 | Line 2–4 | Transformers                     | Transformer Settings|
|:---------:|:-----------------------------------------------|:---------|:---------|:---------|:--------------------------------------------|:-----:|
| p2n       | b1: **Grounded**  b5: **Grounded** · b6: **Grounded** | **3φ+N** | **1φ+N** | **1φ+N** | b3 - b5, b4 - b6           | (1,2,3) |
| mixed     | b1: **Grounded**  b5: **Grounded** · b6: **Grounded** | **3φ**   | **2φ**   | **2φ**   | b3 - b5, b4 - b6 | (1,2,3) |
| p2p       | —                                             | —        | —        | —        | **Not supported**       | —     |

- p2n: phase-to-neutral
- p2p: phase-to-phase
- mixed: phase-to-phase on high voltage bus and phase-to-neutral on low volatge bus

### 5 bus grid, five_bus_1ph_variant_2_mixed_gnd1_case1

In [33]:
net = grids[f"five_bus_1ph_variant_2_mixed_gnd1_case1.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 1.5309634004978534e-08
Max overall va_degree difference: 3.767094764128842e-06
Max overall i_from_ka difference: 3.946472825616665e-09
Max overall i_to_ka difference: 3.946472825616665e-09
Max overall p_from_mw difference: 2.94484660545713e-08
Max overall p_to_mw difference: 2.9448823435362925e-08
Max overall q_from_mvar difference: 1.6946837186050624e-07
Max overall q_to_mvar difference: 1.6946853582028082e-07

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999863 0.999999999863   -0.000001563732   -0.000001544366
      2     0.999999990641 0.999999990641 -120.000000830240 -120.000000817328
      3     1.000000006339 1.000000006339  119.999999260619  119.999999267074
1     0     0.000000000000            NaN    0.000000000000               

### 5 bus grid, five_bus_1ph_variant_2_mixed_gnd1_case2

In [34]:
net = grids[f"five_bus_1ph_variant_2_mixed_gnd1_case2.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 6.284686678847962e-07
Max overall va_degree difference: 7.26834259410225e-05
Max overall i_from_ka difference: 0.00010217550042629017
Max overall i_to_ka difference: 0.00010217550042629017
Max overall p_from_mw difference: 0.000589918106654902
Max overall p_to_mw difference: 0.0005898885202850895
Max overall q_from_mvar difference: 0.00017032197914740657
Max overall q_to_mvar difference: 0.00017032749104601425

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999857 0.999999999857   -0.000001586043   -0.000001555522
      2     0.999999990525 0.999999990582 -120.000000841277 -120.000000822790
      3     1.000000006448 1.000000006392  119.999999249383  119.999999261416
1     0     0.000000000000            NaN    0.000000000000               N

### 5 bus grid, five_bus_1ph_variant_2_mixed_gnd1_case3

In [35]:
net = grids[f"five_bus_1ph_variant_2_mixed_gnd1_case3.pkl"]

# Load pp-multiconductor power flow
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9, MaxIter=100)

# Compare results
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print()
print("bus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print()
print("line results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

Max overall vm_pu difference: 6.619725376744157e-07
Max overall va_degree difference: 7.271335812220059e-05
Max overall i_from_ka difference: 0.00010217548934839463
Max overall i_to_ka difference: 0.00010217548934839463
Max overall p_from_mw difference: 0.0005899180980853264
Max overall p_to_mw difference: 0.0005898885117186919
Max overall q_from_mvar difference: 0.00017031664460682994
Max overall q_to_mvar difference: 0.00017033281644647166

bus results
                  vm_pu_mc      vm_pu_dss      va_degree_mc     va_degree_dss
index phase                                                                  
0     0     0.000000000000 0.000000000000    0.000000000000    0.000000000000
      1     0.999999999857 0.999999999857   -0.000001586043   -0.000001555522
      2     0.999999990525 0.999999990582 -120.000000841277 -120.000000822790
      3     1.000000006448 1.000000006392  119.999999249383  119.999999261416
1     0     0.000000000000            NaN    0.000000000000              